In [16]:
from datasets import load_dataset

ds = load_dataset('aeirya/lct-iranic-lid', 'conf1')

In [17]:
train_test_split = ds["train"].train_test_split(test_size=0.2, seed=42)

test_dev_split = train_test_split["test"].train_test_split(test_size=0.5, seed=42)

ds["train"] = train_test_split["train"]
ds["validation"] = test_dev_split["train"]
ds["test"] = test_dev_split["test"]

ds

DatasetDict({
    train: Dataset({
        features: ['text', 'dataset', 'code', 'script'],
        num_rows: 931872
    })
    validation: Dataset({
        features: ['text', 'dataset', 'code', 'script'],
        num_rows: 116484
    })
    test: Dataset({
        features: ['text', 'dataset', 'code', 'script'],
        num_rows: 116485
    })
})

In [19]:
ds["train"].data.to_pandas()

,text,dataset,code,script
0,"""ماجورکان"" غذاسین پیشیرنده، مدیترانه ای بولگه ...",Flores200,azb,Arab
1,1972-جی ایلده جمع کاراجا موکافاتلا باشلادی. هئ...,mtdata,azb,Arab
2,"او آرتیردی: ""آما گرک اونلاردان توسعه مرحلهسی، ...",Flores200,azb,Arab
3,اولار ایکی ازمایش انجام وردلر کی دی ان ای ای ب...,Flores200,azb,Arab
4,ایش یرین ده هماهنگ اولماخ چوخ مهم دی. شخصی موف...,Flores200,azb,Arab
...,...,...,...,...
1164836,آنها پیشنهاد میکنند که سرمایه اجتماعی باید به ...,oldi,prs,Arab
1164837,عقاب ها، شیر ها گاهی در ستون های شرقی نقاشی می...,oldi,prs,Arab
1164838,اتوم ها در داخل خلاء حرکت کرده، با یکدیگر تعام...,oldi,prs,Arab
1164839,شک گرایی مفرط بدست آوردن اتاراکسیا (یک حالت مت...,oldi,prs,Arab


In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

ds = ds.map(tokenize, batched=True)

Map:   0%|          | 0/116484 [00:00<?, ? examples/s]

In [21]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'dataset', 'code', 'script', 'input_ids', 'attention_mask'],
        num_rows: 931872
    })
    validation: Dataset({
        features: ['text', 'dataset', 'code', 'script', 'input_ids', 'attention_mask'],
        num_rows: 116484
    })
    test: Dataset({
        features: ['text', 'dataset', 'code', 'script', 'input_ids', 'attention_mask'],
        num_rows: 116485
    })
})

In [22]:
from transformers import AutoModelForSequenceClassification


ds = ds.class_encode_column("code")
num_labels = len(ds["train"].features["code"].names)


Casting to class labels:   0%|          | 0/116484 [00:00<?, ? examples/s]

In [23]:
# Training would take ages otherwise
for split_name in ds.keys():
    if 'code' in ds[split_name].features and len(ds[split_name]) > 1:
        ds[split_name] = ds[split_name].train_test_split(
            test_size=0.9,
            stratify_by_column='code'
        )['train']

ds

DatasetDict({
    train: Dataset({
        features: ['text', 'dataset', 'code', 'script', 'input_ids', 'attention_mask'],
        num_rows: 93187
    })
    validation: Dataset({
        features: ['text', 'dataset', 'code', 'script', 'input_ids', 'attention_mask'],
        num_rows: 11648
    })
    test: Dataset({
        features: ['text', 'dataset', 'code', 'script', 'input_ids', 'attention_mask'],
        num_rows: 11648
    })
})

In [24]:
from transformers import AutoModelForSequenceClassification


#ds = ds.class_encode_column("code")
num_labels = len(ds["train"].features["code"].names)

model = AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/xlm-roberta-base",
    num_labels=num_labels
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [25]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [26]:
from transformers import Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }
ds = ds.rename_column("code", "labels")


In [27]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.140934,0.119979,0.960852,0.960624


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5825, training_loss=0.21688375251999228, metrics={'train_runtime': 2642.6648, 'train_samples_per_second': 35.263, 'train_steps_per_second': 2.204, 'total_flos': 6130458011408640.0, 'train_loss': 0.21688375251999228, 'epoch': 1.0})

In [28]:
trainer.evaluate(ds["test"])

{'eval_loss': 0.12750259041786194,
 'eval_accuracy': 0.9604223901098901,
 'eval_f1': 0.9601355453663036,
 'eval_runtime': 92.9998,
 'eval_samples_per_second': 125.248,
 'eval_steps_per_second': 7.828,
 'epoch': 1.0}

In [29]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

classifier("انا تعبان")

[{'label': 'LABEL_14', 'score': 0.5602467656135559}]